# Flow Matching on the 2D Checkerboard

**Phase 2 of the Flow Matching replication** (Lipman et al., 2022, *Flow Matching for Generative Modeling*, arXiv:2210.02747).

**This is the 4×4-grid variant** (16 cells, matching the paper's Figure 4 panel). It is byte-for-byte the Phase-1 pipeline with the *only* change being the target geometry in Section 2 — same plain `TimeConditionedMLP`, same 5000 training steps — so any change in outcome is attributable to the coarser grid alone.

This notebook reuses the Phase-1 (8-Gaussian) pipeline almost unchanged and applies it to the **2D checkerboard**, the toy density used in the paper's Figure 4 (left). The goal is to reproduce the paper's central claims on a *different* target:

- **FM-OT** — Flow Matching with the Optimal-Transport conditional path (eq. 20–23).
- **FM-Diff** — Flow Matching with the Variance-Preserving diffusion path (eq. 18–19).
- **SM-Diff** — Score Matching with the VP path (eq. 42–43) — added later, for the H3 stability comparison.

> **Key point: the ODEs and probability paths do *not* change.**
> The FM-OT and VP paths are defined only by the source `N(0, I)` and a single data endpoint `x1`; they never reference *what* the data distribution is. So the conditional paths, target velocity fields, CFM objective, the network, and all ODE solvers are carried over verbatim. Only the **target sampler** and the **evaluation metrics** change — the checkerboard has no discrete modes, so the mode-based metrics from Phase 1 are replaced by cell/support-based ones.

## Research questions and hypotheses

**Main question.** Do FM-OT, FM-Diff, and SM-Diff, trained under identical architecture and optimisation settings, reproduce the paper's findings on the checkerboard as they did on the 8-Gaussian ring?

**Hypotheses** (adapted from Phase 1; H5 is new and checkerboard-specific):

- **H1 — Support recovery.** At high sampling accuracy, all methods reproduce the full checkerboard support (all "on" cells populated, "off" cells empty).
- **H2 — OT sampling efficiency.** FM-OT preserves distribution quality better at low NFE (on-support rate, MMD², sliced-Wasserstein), because its conditional paths are straight, constant-velocity trajectories.
- **H3 — FM vs SM stability.** FM-Diff trains more stably than SM-Diff on the same VP path (pending SM-Diff).
- **H4 — High-NFE convergence.** Method differences shrink once the ODE is solved accurately.
- **H5 — Pattern emergence (new).** The OT path introduces the checkerboard pattern *earlier* along the generative trajectory than the diffusion path (a direct mirror of Fig. 4, left). We test this by measuring on-support rate at intermediate times `t`.

## 1. Environment and configuration

Standard imports. The pipeline is 2D, so it is light enough to run on CPU, but CUDA is used when available (matching Phase 1).

In [ ]:
import math
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Use CUDA when available; the whole pipeline is tiny so CPU also works.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---- Reproducibility ----
SEED = 42
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("high")  # faster matmuls on recent GPUs

# ---- Model / optimisation config (identical to Phase 1) ----
DATA_DIM              = 2        # 2D toy data
HIDDEN_DIM            = 512      # MLP width  (paper: 512)
NUM_HIDDEN_LAYERS     = 5        # MLP depth  (paper: 5 hidden layers)
LEARNING_RATE         = 5e-4
NUM_TRAINING_STEPS    = 20_000   # longer: fine structure sharpens under the flat loss
BATCH_SIZE            = 2_048    # training mini-batch (fresh samples each step)
VALIDATION_BATCH_SIZE = 4_096
LOG_EVERY             = 100
NUM_PLOT_SAMPLES      = 5_000

# ---- FM-OT config ----
# sigma_min is the residual std at the data endpoint of the OT path (eq. 20).
# Kept identical to Phase 1 so the two experiments are directly comparable.
SIGMA_MIN = 1e-3

print("Model: %d-dim data, %d hidden layers x %d units" % (DATA_DIM, NUM_HIDDEN_LAYERS, HIDDEN_DIM))
print("Training steps:", NUM_TRAINING_STEPS, "| batch:", BATCH_SIZE, "| lr:", LEARNING_RATE)
print("sigma_min:", SIGMA_MIN)

## 2. The checkerboard target distribution

We match the paper's Figure 4 panel: a **4×4 checkerboard** (16 cells, 8 "on"). The domain
$[-L, L]^2$ with $L = 4$ is partitioned into **2×2 cells**, and a cell is **"on"** when its
*cell index* $\left(\lfloor x_1 / s \rfloor + \lfloor x_2 / s \rfloor\right)$ is even, with cell
size $s = 2$. This gives $4 \times 4 = 16$ cells, of which exactly **8 are "on"**.

> **Geometry note (the one easy bug).** With a non-unit cell size, the parity that defines the
> checkerboard is on the cell *index* $\lfloor x / s \rfloor$, **not** on $\lfloor x \rfloor$.
> The support test below divides by `CELL_SIZE` before flooring; forgetting this silently
> describes the wrong grid.

The target density is **uniform on the union of the on-cells**:

$$
q(x) =
\begin{cases}
\dfrac{1}{N_{\text{on}} \cdot A_{\text{cell}}} = \dfrac{1}{8 \cdot 4} = \dfrac{1}{32}, & x \text{ on an on-cell},\\[6pt]
0, & \text{otherwise.}
\end{cases}
$$

Two consequences we use throughout:

1. **Sampling is exact.** Pick an on-cell uniformly (each with probability $1/8$), then a point uniformly inside its $2\times2$ area.
2. **Entropy floor.** Support area is $8 \times 4 = 32$, so the differential entropy is $\log 32 \approx 3.4657$ nats ($\approx 1.733$ nats/dim) — *the same value as the 8×8 grid*, because halving the cell count exactly cancels quadrupling the cell area. It is still an **unreachable** bound for a smooth CNF (the density is discontinuous at cell edges), so model NLL will sit slightly above it and the gap measures boundary leakage.


In [ ]:
# ---- Checkerboard geometry ----
DOMAIN_LIMIT = 4.0     # domain is [-DOMAIN_LIMIT, DOMAIN_LIMIT]^2
CELL_SIZE    = 2.0     # 2x2 cells -> a 4x4 grid, matching the paper's 16-cell panel

# Lower-corner positions of every cell: -4, -2, 0, 2  (4 values per axis -> 16 cells).
_cell_indices = torch.arange(-DOMAIN_LIMIT, DOMAIN_LIMIT, CELL_SIZE)  # [-4, -2, 0, 2]

# Build the list of "on"-cell lower corners. The parity that defines the checkerboard
# is on the *cell index* (corner / CELL_SIZE), not the raw corner coordinate:
#   cell index = corner / CELL_SIZE  ->  {-2, -1, 0, 1};  "on" iff (ix + iy) is even.
_on_corners = []
for cx in _cell_indices:
    for cy in _cell_indices:
        ix = int(round(float(cx) / CELL_SIZE))
        iy = int(round(float(cy) / CELL_SIZE))
        if (ix + iy) % 2 == 0:                  # even -> "on" cell
            _on_corners.append((float(cx), float(cy)))

ON_CELL_CORNERS = torch.tensor(_on_corners, dtype=torch.float32)  # shape (N_on, 2)
NUM_ON_CELLS    = ON_CELL_CORNERS.shape[0]
SUPPORT_AREA    = NUM_ON_CELLS * (CELL_SIZE ** 2)
ENTROPY_FLOOR   = math.log(SUPPORT_AREA)         # differential entropy = -log density

print("On-cells:", NUM_ON_CELLS, "| support area:", SUPPORT_AREA)
print("Entropy floor (nats):    ", ENTROPY_FLOOR)
print("Entropy floor (nats/dim):", ENTROPY_FLOOR / DATA_DIM)

In [ ]:
def on_checkerboard_support(x: torch.Tensor) -> torch.Tensor:
    """
    Boolean mask: True where x lies on an "on" cell of the checkerboard.

    A point is on-support iff it is inside the domain AND the integer cell it
    falls into, (floor(x1), floor(x2)), satisfies (floor(x1) + floor(x2)) even.
    """
    inside = (x.abs() <= DOMAIN_LIMIT).all(dim=-1)          # within [-L, L]^2
    cell   = torch.floor(x / CELL_SIZE)                     # cell index for arbitrary cell size
    parity_even = ((cell[:, 0] + cell[:, 1]) % 2 == 0)      # "on" cells have even parity
    return inside & parity_even

In [ ]:
def sample_checkerboard(n_samples: int, device: torch.device) -> torch.Tensor:
    """
    Exact sampler for the uniform checkerboard density.

    Step 1: choose an on-cell uniformly at random (each of the 32 with prob 1/32).
    Step 2: place the point uniformly inside that unit cell.
    """
    corners = ON_CELL_CORNERS.to(device)                    # (N_on, 2) lower corners
    # Uniformly pick one on-cell per sample.
    choice = torch.randint(0, NUM_ON_CELLS, (n_samples,), device=device)
    chosen_corner = corners[choice]                         # (n_samples, 2)
    # Uniform offset within the unit cell.
    offset = torch.rand(n_samples, 2, device=device) * CELL_SIZE
    return chosen_corner + offset

In [ ]:
def checkerboard_log_density(x: torch.Tensor) -> torch.Tensor:
    """
    Exact analytical log-density log q(x) of the uniform checkerboard.

    log q(x) = -log(support area) on-support, and -inf off-support.
    (The -inf branch is only used for reference; the model-NLL evaluation later
    always feeds on-support data samples.)
    """
    on_support = on_checkerboard_support(x)
    log_density = torch.full(
        (x.shape[0],), -math.log(SUPPORT_AREA), device=x.device, dtype=x.dtype
    )
    log_density = torch.where(
        on_support, log_density, torch.full_like(log_density, float("-inf"))
    )
    return log_density

In [ ]:
# ---- Sanity checks on the target definition ----
_probe = sample_checkerboard(10_000, device)

print("All sampled points on-support:", on_checkerboard_support(_probe).all().item())
print("Sampled log-density (should equal -log 32 = %.4f):" % (-math.log(SUPPORT_AREA)),
      checkerboard_log_density(_probe).mean().item())

# An off-support point (a cell centre with odd parity) must be flagged off-support.
_off = torch.tensor([[-1.0, 1.0]], device=device)  # cell index (-1, 0), parity odd -> off
print("Odd-parity point on-support (should be False):", on_checkerboard_support(_off).item())

### 2.1 Visualising the target

A helper to draw the on-cell boundaries is reused in every later plot (it replaces the
mode-centre crosses from Phase 1).

In [ ]:
def draw_checkerboard_cells(ax, color="tab:orange", lw=1.0, alpha=0.5):
    """Overlay the outline of every "on" cell so plots show the target support."""
    from matplotlib.patches import Rectangle
    for cx, cy in ON_CELL_CORNERS.tolist():
        ax.add_patch(Rectangle((cx, cy), CELL_SIZE, CELL_SIZE,
                               fill=False, edgecolor=color, lw=lw, alpha=alpha))

# Draw a batch of target samples with the cell overlay.
_vis = sample_checkerboard(NUM_PLOT_SAMPLES * 2, device).cpu()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(_vis[:, 0], _vis[:, 1], s=3, alpha=0.35)
draw_checkerboard_cells(ax)
ax.set_title("Checkerboard Target Distribution")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.set_xlim(-DOMAIN_LIMIT - 0.5, DOMAIN_LIMIT + 0.5)
ax.set_ylim(-DOMAIN_LIMIT - 0.5, DOMAIN_LIMIT + 0.5)
ax.set_aspect("equal"); ax.grid(alpha=0.15)
plt.show()

## 3. FM-OT conditional probability path

Unchanged from Phase 1 — reproduced here for a self-contained notebook and re-validated on the new target.

The source is $p_0 = \mathcal{N}(0, I)$. For a data point $x_1$, the OT conditional flow (paper eq. 22) is the straight line

$$
\psi_t(x_0) = \bigl(1 - (1 - \sigma_{\min})\,t\bigr)\, x_0 + t\, x_1,
$$

and the corresponding conditional target velocity (eq. 23) is **constant in $t$**:

$$
u_t(x \mid x_1) = x_1 - (1 - \sigma_{\min})\, x_0.
$$

The Conditional Flow Matching (CFM) loss regresses a network $v_\theta(t, x)$ onto $u_t$.

In [ ]:
def sample_source(n_samples: int, device: torch.device) -> torch.Tensor:
    """Draw from the standard 2D Gaussian source p0 = N(0, I)."""
    return torch.randn(n_samples, DATA_DIM, device=device)

In [ ]:
def fm_ot_state(x0, x1, t, sigma_min: float = SIGMA_MIN):
    """
    Point on the OT conditional path at time t (paper eq. 22):
        psi_t(x0) = (1 - (1 - sigma_min) t) * x0 + t * x1
    t has shape (batch, 1) so it broadcasts over the 2 coordinates.
    """
    coefficient = 1.0 - (1.0 - sigma_min) * t
    return coefficient * x0 + t * x1

In [ ]:
def fm_ot_target_velocity(x0, x1, sigma_min: float = SIGMA_MIN):
    """
    Conditional target velocity for the OT path (paper eq. 23):
        u = x1 - (1 - sigma_min) * x0
    Note: independent of t -> constant-direction, constant-speed trajectories.
    """
    return x1 - (1.0 - sigma_min) * x0

In [ ]:
def sample_fm_ot_batch(batch_size: int, device: torch.device, sigma_min: float = SIGMA_MIN):
    """
    Build one simulation-free FM-OT training mini-batch:
      x0 ~ N(0, I),  x1 ~ checkerboard,  t ~ U[0, 1],
      xt = psi_t(x0),  ut = conditional target velocity.
    Fresh samples every call -> the loss is an unbiased Monte-Carlo estimate of CFM.
    """
    x0 = sample_source(batch_size, device)
    x1 = sample_checkerboard(batch_size, device)
    t  = torch.rand(batch_size, 1, device=device)
    xt = fm_ot_state(x0, x1, t, sigma_min)
    ut = fm_ot_target_velocity(x0, x1, sigma_min)
    return x0, x1, t, xt, ut

In [ ]:
# ---- Validate the path endpoints and the analytic velocity ----
x0, x1, t, xt, ut = sample_fm_ot_batch(BATCH_SIZE, device)

# At t=0 the state must equal x0; at t=1 it must equal x1 (up to sigma_min * x0).
xt_start = fm_ot_state(x0, x1, torch.zeros_like(t))
xt_end   = fm_ot_state(x0, x1, torch.ones_like(t))
print("t=0 state == x0:", torch.allclose(xt_start, x0, atol=1e-6))
print("t=1 state ~= x1:", torch.allclose(xt_end, x1, atol=1e-2))  # sigma_min residual

# The analytic velocity must match a finite-difference derivative of psi_t.
h = 1e-4
fd = (fm_ot_state(x0, x1, t + h) - fm_ot_state(x0, x1, t - h)) / (2 * h)
print("Max |analytic u - finite-diff dpsi/dt|:", (ut - fd).abs().max().item())

### 3.1 How the conditional path morphs noise into the checkerboard

This plot is model-free: it just pushes source samples along $\psi_t$ toward fresh checkerboard
targets at several times. It documents *what the network is asked to learn to reverse-engineer*,
and previews H5 — watch how late the pattern appears along the path.

In [ ]:
path_times = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, len(path_times), figsize=(19, 4), sharex=True, sharey=True)

x0_vis = sample_source(NUM_PLOT_SAMPLES, device)
x1_vis = sample_checkerboard(NUM_PLOT_SAMPLES, device)

for ax, tv in zip(axes, path_times):
    t_col = torch.full((NUM_PLOT_SAMPLES, 1), tv, device=device)
    state = fm_ot_state(x0_vis, x1_vis, t_col).cpu()
    ax.scatter(state[:, 0], state[:, 1], s=3, alpha=0.3)
    ax.set_title(f"t = {tv:.2f}")
    ax.set_xlabel("Coordinate 1")
    ax.set_xlim(-6, 6); ax.set_ylim(-6, 6); ax.set_aspect("equal"); ax.grid(alpha=0.15)
axes[0].set_ylabel("Coordinate 2")
fig.suptitle("FM-OT Conditional Probability Path (noise -> checkerboard)", fontsize=14)
plt.tight_layout(); plt.show()

## 4. Neural vector field (Fourier-feature MLP)

The paper only specifies "an MLP with 5 layers of 512 neurons" (Appendix E) for the 2D toys and
is silent on the input representation. A plain-coordinate MLP hits the CFM loss floor yet smooths
away the checkerboard's sharp cell edges — **spectral bias** against a high-frequency target that
the smooth 8-Gaussian ring never exposed. We therefore feed the network **random Fourier features**
of both the coordinate $x$ and the time $t$; this only changes the input representation, not the
method, paths, loss, or the $v_\theta(x, t)$ signature.

Knobs (tuned on the finer 8×8 grid, kept here for consistency):

- `x_num_freq = 128`, `x_scale = 2.2` — spatial frequency basis; `x_scale` is the main dial for edge
  sharpness (too high → speckle in the gaps, too low → soft edges). On this coarser 4×4 grid the
  target is *half the spatial frequency*, so `x_scale = 2.2` may be higher than necessary — if the
  gaps come out speckled, try lowering toward `1.0–1.5`.
- 5×512 SiLU trunk, unchanged. The input layer is wider now, so the parameter count is **larger than
  the ~1.05M plain-MLP figure** — read the printed count below.

Trained for 20k steps with cosine LR decay: the loss curve flattens early, but the high-frequency
component keeps improving well past that under the flat line, which is why the longer schedule matters.


In [ ]:
# --- Fourier-feature model (replaces the plain MLP for the checkerboard) ---
# A plain MLP on raw coordinates suffers from spectral bias: it hits the CFM loss floor
# while smoothing away the sharp cell edges. Random Fourier features give the network the
# high-frequency basis it needs to resolve those edges. The method/paths are unchanged;
# only the input representation to v_theta changes.

class FourierFeatures(nn.Module):
    """Random Fourier features: v -> [sin(2*pi*v@B), cos(2*pi*v@B)] with a fixed B.
    'scale' sets the frequency band -> larger scale resolves sharper spatial structure."""
    def __init__(self, in_dim, num_frequencies, scale):
        super().__init__()
        # Fixed (non-trainable) Gaussian frequency matrix, stored as a buffer so it
        # moves with .to(device) and is saved in the checkpoint.
        self.register_buffer("B", torch.randn(in_dim, num_frequencies) * scale)
    def forward(self, v):
        proj = 2 * math.pi * (v @ self.B)              # (batch, num_frequencies)
        return torch.cat([proj.sin(), proj.cos()], dim=-1)


class FourierMLP(nn.Module):
    """Same 5x512 SiLU MLP, but the input is [x, FF(x), t, FF(t)].
    Signature v_theta(x, t) is identical, so nothing downstream changes."""
    def __init__(self, data_dim, hidden_dim, num_hidden_layers,
                 x_num_freq=128, x_scale=2.2, t_num_freq=32, t_scale=5.0):
        super().__init__()
        self.x_features = FourierFeatures(data_dim, x_num_freq, x_scale)
        self.t_features = FourierFeatures(1,        t_num_freq, t_scale)
        in_dim = data_dim + 2 * x_num_freq + 1 + 2 * t_num_freq   # raw x + FF(x) + raw t + FF(t)
        layers = [nn.Linear(in_dim, hidden_dim), nn.SiLU()]
        for _ in range(num_hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.SiLU()]
        layers += [nn.Linear(hidden_dim, data_dim)]
        self.network = nn.Sequential(*layers)
    def forward(self, x, t):
        feats = torch.cat([x, self.x_features(x), t, self.t_features(t)], dim=-1)
        return self.network(feats)

In [ ]:
# Tuned checkerboard model: 128 spatial frequencies at scale 2.2 (see notes).
fm_ot_model = FourierMLP(DATA_DIM, HIDDEN_DIM, NUM_HIDDEN_LAYERS,
                         x_num_freq=128, x_scale=2.2).to(device)

n_params = sum(p.numel() for p in fm_ot_model.parameters() if p.requires_grad)
print(fm_ot_model)
print("Trainable parameters:", f"{n_params:,}")

# Quick forward-pass check.
with torch.no_grad():
    _v = fm_ot_model(xt[:128], t[:128])
print("Forward output shape:", tuple(_v.shape), "| all finite:", torch.isfinite(_v).all().item())

## 5. Training objective and fixed validation batch

The CFM loss is the mean squared velocity error, summed over coordinates:

$$
\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t,\,x_0,\,x_1}\bigl\| v_\theta(x_t, t) - u_t(x_t \mid x_1) \bigr\|^2 .
$$

A **fixed** validation batch (frozen once) gives a stable curve to watch, exactly as in Phase 1.

In [ ]:
def velocity_matching_loss(predicted_velocity, target_velocity):
    """Mean over the batch of the squared L2 velocity error (summed over coordinates)."""
    return (predicted_velocity - target_velocity).square().sum(dim=-1).mean()

# Frozen validation batch (same generator, drawn once).
torch.manual_seed(SEED + 1)
val_x0, val_x1, val_t, val_xt, val_ut = sample_fm_ot_batch(VALIDATION_BATCH_SIZE, device)
torch.manual_seed(SEED)  # restore
print("Validation batch:", tuple(val_xt.shape))

## 6. FM-OT training

Simulation-free training: every step draws a **fresh** mini-batch (no fixed dataset), so each
gradient is an unbiased estimate of the CFM objective. 5000 steps, Adam, lr 5e-4 — identical to Phase 1.

> The training loop is written out explicitly below (rather than hidden in a helper) so the
> per-step logging matches Phase 1 and every line is visible for analysis.

In [ ]:
optimizer = torch.optim.Adam(fm_ot_model.parameters(), lr=LEARNING_RATE)
# Cosine decay lets the model settle into the fine cell structure instead of
# jittering at the peak lr once the coarse field is learned.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_TRAINING_STEPS)
history = {"step": [], "train": [], "val": []}
start = time.time()

for step in range(1, NUM_TRAINING_STEPS + 1):
    # ---- one optimisation step on a fresh simulation-free batch ----
    fm_ot_model.train()
    _, _, t_b, xt_b, ut_b = sample_fm_ot_batch(BATCH_SIZE, device)
    pred = fm_ot_model(xt_b, t_b)
    loss = velocity_matching_loss(pred, ut_b)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    scheduler.step()

    # ---- periodic logging on the frozen validation batch ----
    if step % LOG_EVERY == 0 or step == 1:
        fm_ot_model.eval()
        with torch.no_grad():
            val_loss = velocity_matching_loss(fm_ot_model(val_xt, val_t), val_ut)
        history["step"].append(step)
        history["train"].append(loss.item())
        history["val"].append(val_loss.item())
        mem = (torch.cuda.max_memory_allocated() / 1e6) if device.type == "cuda" else 0.0
        print(f"Step {step:5d}/{NUM_TRAINING_STEPS} | "
              f"Train {loss.item():.4f} | Val {val_loss.item():.4f} | "
              f"GPU {mem:6.1f} MB | {time.time()-start:5.1f} s")

print("Training completed.")

In [ ]:
# ---- Loss curves ----
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(history["step"], history["train"], label="Training loss")
ax.plot(history["step"], history["val"],   label="Validation loss")
ax.set_yscale("log")
ax.set_xlabel("Optimisation step"); ax.set_ylabel("Squared velocity error")
ax.set_title("FM-OT Conditional Flow Matching Loss (checkerboard)")
ax.legend(); ax.grid(alpha=0.2)
plt.show()

## 7. Save the FM-OT checkpoint

Saved so the sampling/metric cells (and the FM-Diff comparison) can reload without retraining.

In [ ]:
CHECKPOINT_DIR = Path("outputs/checkers")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
torch.save({"model_state": fm_ot_model.state_dict(),
            "config": {"DATA_DIM": DATA_DIM, "HIDDEN_DIM": HIDDEN_DIM,
                       "NUM_HIDDEN_LAYERS": NUM_HIDDEN_LAYERS, "SIGMA_MIN": SIGMA_MIN}},
           CHECKPOINT_DIR / "fm_ot_checkers.pt")
print("Saved:", (CHECKPOINT_DIR / "fm_ot_checkers.pt").resolve())

## 8. ODE samplers (Euler, midpoint, adaptive dopri5)

Sampling integrates $\dot{x} = v_\theta(x, t)$ from $t=0$ (noise) to $t=1$ (data). All three
solvers are carried over from Phase 1 unchanged, including the NFE-counting wrapper. The FM-OT
path is integrated over the full $[0, 1]$; the VP path (FM-Diff, later) will need the
$t = 1-\varepsilon$ endpoint because its velocity is singular at exactly $t=1$.

In [ ]:
from torchdiffeq import odeint

class NeuralODEFunction(nn.Module):
    """Adapts v_theta to the torchdiffeq signature f(t, x) and counts NFE."""
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.nfe = 0
    def reset_nfe(self):
        self.nfe = 0
    def forward(self, t, x):
        self.nfe += 1
        t_col = torch.ones(x.shape[0], 1, device=x.device, dtype=x.dtype) * t
        return self.model(x, t_col)

In [ ]:
@torch.no_grad()
def euler_integrate(model, x0, num_steps, t_start=0.0, t_end=1.0):
    """Fixed-step explicit Euler. NFE = num_steps."""
    model.eval()
    x = x0.clone()
    dt = (t_end - t_start) / num_steps
    for k in range(num_steps):
        t = t_start + k * dt
        t_col = torch.full((x.shape[0], 1), t, device=x.device, dtype=x.dtype)
        x = x + dt * model(x, t_col)
    return x  # NFE == num_steps

In [ ]:
@torch.no_grad()
def midpoint_integrate(model, x0, num_steps, t_start=0.0, t_end=1.0):
    """
    Fixed-step midpoint (RK2). Two model evals per step -> NFE = 2 * num_steps.
    Matches the low-cost solver used in the paper's Fig. 4 (right).
    """
    model.eval()
    x = x0.clone()
    dt = (t_end - t_start) / num_steps
    for k in range(num_steps):
        t = t_start + k * dt
        t_col  = torch.full((x.shape[0], 1), t,          device=x.device, dtype=x.dtype)
        t_half = torch.full((x.shape[0], 1), t + dt / 2, device=x.device, dtype=x.dtype)
        k1 = model(x, t_col)
        k2 = model(x + 0.5 * dt * k1, t_half)
        x = x + dt * k2
    return x  # NFE == 2 * num_steps

In [ ]:
DOPRI5_ATOL = 1e-5
DOPRI5_RTOL = 1e-5

@torch.no_grad()
def dopri5_integrate(model, x0, t_start=0.0, t_end=1.0, atol=DOPRI5_ATOL, rtol=DOPRI5_RTOL):
    """
    Adaptive Dormand-Prince solve from t_start to t_end (paper's default at atol=rtol=1e-5).
    Returns (final_state, nfe).
    """
    model.eval()
    ode_fn = NeuralODEFunction(model).to(x0.device)
    ode_fn.reset_nfe()
    times = torch.tensor([t_start, t_end], device=x0.device, dtype=x0.dtype)
    solution = odeint(ode_fn, x0, times, method="dopri5", atol=atol, rtol=rtol)
    return solution[-1], ode_fn.nfe

## 9. Generate samples and qualitative comparison

A fixed evaluation set of source samples is reused across solvers so differences are not due to
randomness. We generate with the adaptive solver and compare against the analytical target, with
the cell overlay. Quantitative evaluation (on-support rate, cell-KL, MMD², sliced-Wasserstein,
model NLL) begins in the next batch.

In [ ]:
# Fixed evaluation noise (reused by every solver / method).
NUM_EVAL_SAMPLES = 10_000
torch.manual_seed(SEED + 2)
eval_source = sample_source(NUM_EVAL_SAMPLES, device)
torch.manual_seed(SEED)

# Reference target sample (for side-by-side plots and later metrics).
target_reference = sample_checkerboard(NUM_EVAL_SAMPLES, device)

# Generate with the adaptive solver over the full [0, 1] interval.
t0 = time.time()
fm_ot_generated, fm_ot_nfe = dopri5_integrate(fm_ot_model, eval_source, t_start=0.0, t_end=1.0)
print(f"FM-OT dopri5: NFE={fm_ot_nfe} | {time.time()-t0:.2f}s | "
      f"all finite: {torch.isfinite(fm_ot_generated).all().item()}")

In [ ]:
# Side-by-side: analytical target vs FM-OT generated samples.
tgt = target_reference.cpu(); gen = fm_ot_generated.cpu()
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), sharex=True, sharey=True)

for ax, data, title in [(axes[0], tgt, "Analytical Target"),
                        (axes[1], gen, f"FM-OT  (dopri5, NFE={fm_ot_nfe})")]:
    ax.scatter(data[:, 0], data[:, 1], s=3, alpha=0.3)
    draw_checkerboard_cells(ax)
    ax.set_title(title); ax.set_xlabel("Coordinate 1")
    ax.set_xlim(-DOMAIN_LIMIT - 0.5, DOMAIN_LIMIT + 0.5)
    ax.set_ylim(-DOMAIN_LIMIT - 0.5, DOMAIN_LIMIT + 0.5)
    ax.set_aspect("equal"); ax.grid(alpha=0.15)
axes[0].set_ylabel("Coordinate 2")
fig.suptitle("FM-OT Generation on the Checkerboard", fontsize=14)
plt.tight_layout(); plt.show()

---
### Next batch: checkerboard metric suite

From here the pipeline diverges most from Phase 1. The next cells (delivered one at a time) will build:

1. **On-support rate** — fraction of generated points on on-cells (replaces "high-quality rate").
2. **Cell-uniformity KL** — empirical mass over the 32 on-cells vs uniform (replaces mode-KL).
3. **Cells covered** — on-cells above a mass threshold, out of 32 (replaces discovered modes).
4. **MMD²** and **sliced-Wasserstein** — carried over unchanged (sample-vs-sample).
5. **Model NLL** via change-of-variables — carried over, interpreted against the $\log 32$ floor (unreachable bound).
6. **H5 pattern-emergence** experiment — on-support rate vs intermediate time $t$, FM-OT vs FM-Diff.

We'll add each with its markdown explanation so you can diff it against the Phase-1 equivalent.